# Competitive race (A — qualification)

Before you start working with this notebook, remember to:

* Specify the correct list of teams with colors.
* Specify the correct path to student designs (directory) and to student roster (file). Ensure that the student roster is a JSON file with an array of dictionaries with keys `netid`, `first_name`, `last_name`, `dp4_partner`, and `dp4_team`.
* Ensure that `imagemagick` is installed (e.g., `conda install imagemagick`) for the command-line utility `convert`.

You may also need to activate a conda environment with `python-control`, depending on whether or not students expect that module to be available.

Specify list of teams with colors.

In [ ]:
teams = {
    'casey': [0.502, 0., 0.502],
    'damian': [1.0, 0.84, 0.],
    'luke': [0.114, 0.788, 0.42],
    'mj': [1.0, 0.373, 0.02],
}

Specify path to student designs and to student roster.

In [ ]:
path_designs = 'do_not_commit/test_designs'
path_roster = 'do_not_commit/test_students.json'

Import modules and configure the notebook.

In [ ]:
import os
from datetime import datetime
import numpy as np
import secrets
import json
import shutil
import subprocess
from pathlib import Path
import ae353_drone

Prevent students from importing `ae353_drone` in their own code.

In [ ]:
import sys
sys.modules['ae353_drone'] = None

Treat `RuntimeWarnings` as errors so that student code does not completely break the simulator (e.g., making it so collision checking stops working and drones fly through the ground).

In [ ]:
import warnings
warnings.filterwarnings('error', category=RuntimeWarning)

Create and print seed so it is possible to reproduce the results.

In [ ]:
seed = secrets.randbits(32)
print(seed)

Create simulator.

In [ ]:
simulator = ae353_drone.Simulator(seed=seed)

Copy student submissions and student roster.

In [ ]:
# Get string with current date and time
datetimestr = datetime.now().strftime('%Y%m%dT%H%M%S')

# Copy student submissions
srcdir_designs = f'{datetimestr}-designs'
results = shutil.copytree(
    Path(path_designs),
    srcdir_designs,
)

# Copy student roster
filename_students = f'{datetimestr}-students.json'
results = shutil.copyfile(
    Path(path_roster),
    filename_students,
)

Load student roster.

In [ ]:
with open(filename_students, 'r') as infile:
    students = json.load(infile)

def get_student(students, netid):
    for student in students:
        if student['netid'] == netid:
            return student
    return None

def get_partners(students, student):
    partner_netids = np.array(student['dp4_partner']).flatten().tolist()
    partner_students = []
    for netid in partner_netids:
        partner_students.append(get_student(students, netid))
    return partner_students

Make sure all files in source directory have lower-case names.

In [ ]:
srcdir = srcdir_designs
for file in os.listdir(srcdir):
    os.rename(os.path.join(srcdir, file), os.path.join(srcdir, file.lower()))

Make sure all PNG files in source directory really are PNG files.

In [ ]:
srcdir = srcdir_designs
template_image = 'question_mark.png'
for file in os.listdir(srcdir):
    if file.endswith('.png'):
        completed_process = subprocess.run([
                    'convert',
                    os.path.join(srcdir, file),
                    os.path.join(srcdir, file),
                ], capture_output=True)
        if completed_process.returncode != 0:
            print(f'   ** FAILED on {file} (returncode: {completed_process.returncode}), replacing with template')
            shutil.copyfile(template_image, os.path.join(srcdir, file))

Look for and move submissions with names that do not have the form `netid.py`.

In [ ]:
srcdir = srcdir_designs
for file in os.listdir(srcdir):
    if file.endswith('.py'):
        netid = file.removesuffix('.py')
        student = get_student(students, netid)
        if student is None:
            print(f'  ** BAD CODE NAME - {file} moved to "bad-code-name-{file}"')
            src = os.path.join(srcdir, file)
            dst = os.path.join(srcdir, f'bad-code-name-{file}')
            shutil.move(src, dst)

Look for and move duplicate submissions.

In [ ]:
netids_to_email = []
groups = []
srcdir = srcdir_designs
for file in os.listdir(srcdir):
    if file.endswith('.py'):
        netid = file.removesuffix('.py')
        student = get_student(students, netid)
        if student is None:
            continue
        group = student['dp4_group_name']
        if group in groups:
            name = f'{student["first_name"]} {student["last_name"]}'
            print(f'  ** DUPLICATE SUBMISSION by {name} for {group}\n       (moved to "duplicate-{file}")')
            netids_to_email.append(student['netid'] + '@illinois.edu')
            partners = get_partners(students, student)
            for partner in partners:
                netids_to_email.append(partner['netid'] + '@illinois.edu')
            src = os.path.join(srcdir, file)
            dst = os.path.join(srcdir, f'duplicate-{file}')
            shutil.move(src, dst)
        groups.append(group)

if len(netids_to_email) > 0:
    print(f'\nSTUDENTS TO EMAIL ({len(netids_to_email)}):\n')
    print(' ' + ', '.join(netids_to_email))

Load drones from source directory, overriding the maximum allowable number.

In [ ]:
simulator.clear_drones()
failures = simulator.load_drones(srcdir_designs, no_max_num_drones=True)

List disqualified drones.

In [ ]:
netids_to_email = []
print(f'DISQUALIFIED ({len(failures)}):\n')
for failure in failures:
    if failure.startswith('bad-code-name'):
        continue
    
    if failure.startswith('duplicate'):
        continue
    
    student = get_student(students, failure)
    if student is None:
        name = ''
    else:
        student['dp4_status'] = 'disqualified'
        name = f'{student["first_name"]} {student["last_name"]}'
        netids_to_email.append(student['netid'] + '@illinois.edu')
        partners = get_partners(students, student)
        for partner in partners:
            partner['dp4_status'] = 'disqualified'
            name += f' and {partner["first_name"]} {partner["last_name"]}'
            netids_to_email.append(partner['netid'] + '@illinois.edu')
    print(f' {failure:20s} : {name}')

if len(netids_to_email) > 0:
    print(f'\nSTUDENTS TO EMAIL ({len(netids_to_email)}):\n')
    print(' ' + ', '.join(netids_to_email))

List qualified drones.

In [ ]:
print(f'QUALIFIED ({len(simulator.drones)}):\n')
for drone in simulator.drones:
    student = get_student(students, drone['name'])
    if student is None:
        raise Exception(f'could not find student for this drone name: {drone["name"]}')
    student['dp4_status'] = 'qualified'
    name = f'{student["first_name"]} {student["last_name"]}'
    partners = get_partners(students, student)
    for partner in partners:
        partner['dp4_status'] = 'qualified'
        name += f' and {partner["first_name"]} {partner["last_name"]}'
    print(f' {drone["name"]:15s} : {student["dp4_team"]:15s} : {name}')

List non-submissions.

In [ ]:
netids_to_email = []

print(f'NON-SUBMISSIONS:\n')
for student in students:
    if not 'dp4_status' in student:
        student['dp4_status'] = 'did not submit'
        name = f'{student["first_name"]} {student["last_name"]}'
        netids_to_email.append(student['netid'] + '@illinois.edu')
        partners = get_partners(students, student)
        for partner in partners:
            partner['dp4_status'] = 'did not submit'
            name += f' and {partner["first_name"]} {partner["last_name"]}'
            netids_to_email.append(partner['netid'] + '@illinois.edu')
        print(f' {name}')

if len(netids_to_email) > 0:
    print(f'\nSTUDENTS TO EMAIL ({len(netids_to_email)}):\n')
    print(' ' + ', '.join(netids_to_email))

Save results of qualification to file.

In [ ]:
# Student roster (updated with qualification failures)
with open(f'{datetimestr}-A-students.json', 'w') as outfile:
    json.dump(students, outfile, indent=4)

# List of students who qualified
qualified = [drone['name'] for drone in simulator.drones]
with open(f'{datetimestr}-A-qualified.json', 'w') as outfile:
    json.dump(qualified, outfile, indent=4)

# Information about race
race_information_path = Path('race-information.json')
if race_information_path.exists():
    print(f'WARNING: race information file exists (moved to race-information-before-{datetimestr}.json)')
    shutil.move(
        race_information_path,
        Path(f'race-information-before-{datetimestr}.json'),
    )
with open(race_information_path, 'w') as outfile:
    json.dump(
        {
            'datetimestr': datetimestr,
            'seed-A': seed,
            'teams': teams,
        },
        outfile,
        indent=4,
    )